# v3 — Multi-model well location recommendation (Jupyter Notebook)

Этот ноутбук делает **полный multi-model pipeline** для задачи выбора перспективного места для новой скважины.

Что внутри:

1. Очистка и агрегация данных  
2. `well_base` — итоговая таблица скважин  
3. Leakage-free target  
4. Spatial clusters для честной spatial CV  
5. Neighbor features  
6. Сравнение моделей:
   - KNN Spatial Baseline
   - Logistic Regression
   - Random Forest
   - LightGBM
   - XGBoost
   - CatBoost Classifier
   - CatBoost Regressor as Ranker
7. Финальные модели на всех скважинах  
8. Генерация кандидатных пустых точек  
9. Filtering module  
10. Ranking module  
11. Карта рекомендаций  
12. Сохранение результатов

> Ноутбук написан **под Jupyter Notebook** и запускается **по ячейкам сверху вниз**.

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn catboost lightgbm xgboost scipy openpyxl joblib

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import joblib
from scipy.stats import gaussian_kde

from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.neighbors import NearestNeighbors

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

sns.set(style="whitegrid")
RANDOM_STATE = 42
EARTH_RADIUS_KM = 6371.0

In [ ]:
file_path = r"D:\User\Downloads\alltime_data.csv" 
# file_path = "alltime_data.xlsx"

## 1. Загрузка и базовая очистка

In [ ]:
file_path = Path(file_path)

if file_path.suffix.lower() == ".csv":
    df_raw = pd.read_csv(file_path)
elif file_path.suffix.lower() in [".xlsx", ".xls"]:
    df_raw = pd.read_excel(file_path)
else:
    raise ValueError("Файл должен быть CSV или Excel")

print("Данные загружены:", df_raw.shape)
df_raw.head()

In [ ]:
expected_columns = [
    "well_id","date","horizon","well_category","block","x","y","coord_system",
    "oil_mass_rate","work_day","oil_mass","wcut_rate","liq_vol","liq_vol_rate"
]

missing_cols = [c for c in expected_columns if c not in df_raw.columns]
print("Не хватает колонок:", missing_cols)

In [ ]:
df = df_raw.copy()

df["date"] = pd.to_datetime(df["date"], errors="coerce")

numeric_cols = [
    "x","y","oil_mass_rate","work_day","oil_mass","wcut_rate","liq_vol","liq_vol_rate"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

cat_cols = ["well_id","horizon","well_category","block","coord_system"]
for col in cat_cols:
    df[col] = df[col].astype("string").str.strip()

df["horizon"] = df["horizon"].str.replace(r"\.0$", "", regex=True)

for col in ["horizon","well_category","block","coord_system"]:
    df[col] = df[col].replace(["", "<NA>", "nan", "None", None], pd.NA)

df["date_month"] = df["date"].dt.to_period("M").dt.to_timestamp()
df["year"] = df["date_month"].dt.year
df["month"] = df["date_month"].dt.month
df["quarter"] = df["date_month"].dt.quarter
df["year_month"] = df["date_month"].dt.to_period("M").astype(str)

df = df.drop_duplicates().copy()

print("После очистки:", df.shape)
df.head()

## 2. Вспомогательные функции

In [ ]:
def mode_val(series):
    mode = series.mode(dropna=True)
    if len(mode) > 0:
        return mode.iloc[0]
    return np.nan

def safe_sum(series):
    return series.sum(min_count=1)

def prepare_coords_rad(df_in):
    # haversine: порядок latitude, longitude
    return np.radians(df_in[["y", "x"]].to_numpy())

def add_spatial_clusters(df_in, n_clusters=5, random_state=42):
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
    labels = km.fit_predict(df_in[["x", "y"]])
    out = df_in.copy()
    out["spatial_cluster"] = labels
    return out, km

## 3. Агрегация до уровня скважины

In [ ]:
df_wells = (
    df
    .groupby("well_id", observed=True)
    .agg(
        x=("x", "mean"),
        y=("y", "mean"),
        horizon=("horizon", mode_val),
        well_category=("well_category", mode_val),
        block=("block", mode_val),
        coord_system=("coord_system", mode_val),

        oil_mass_total=("oil_mass", safe_sum),
        liq_vol_total=("liq_vol", safe_sum),
        work_day_total=("work_day", safe_sum),

        oil_mass_rate_mean=("oil_mass_rate", "mean"),
        liq_vol_rate_mean=("liq_vol_rate", "mean"),
        wcut_rate_mean=("wcut_rate", "mean"),

        months_active=("date_month", "nunique"),
        first_date=("date_month", "min"),
        last_date=("date_month", "max")
    )
    .reset_index()
)

print("Скважин после агрегации:", df_wells.shape)
df_wells.head(3)

## 4. Формирование `well_base`

In [ ]:
wells = df_wells.copy()

# Координаты обязательны
wells = wells.dropna(subset=["x", "y"]).copy()

# Если в данных есть WGS 84 — оставляем его
if "coord_system" in wells.columns:
    has_wgs = wells["coord_system"].astype("string").str.contains("WGS", case=False, na=False).any()
    if has_wgs:
        wells = wells[wells["coord_system"].astype("string").str.contains("WGS", case=False, na=False)].copy()

# Границы координат
wells = wells[
    (wells["x"].between(-180, 180)) &
    (wells["y"].between(-90, 90))
].copy()

# Минимальная история
wells = wells[wells["months_active"] >= 12].copy()

# Производные
wells["oil_per_month"] = wells["oil_mass_total"] / wells["months_active"].replace(0, np.nan)
wells["liq_per_month"] = wells["liq_vol_total"] / wells["months_active"].replace(0, np.nan)

# Категории
for col in ["horizon", "well_category", "block"]:
    wells[col] = wells[col].astype("string").fillna("unknown")

# Убираем строки без ключевых метрик
wells = wells.dropna(subset=[
    "oil_per_month", "oil_mass_rate_mean", "wcut_rate_mean", "x", "y"
]).copy()

# Убираем сильные выбросы по среднему дебиту
oil_rate_q995 = wells["oil_mass_rate_mean"].quantile(0.995)
wells = wells[wells["oil_mass_rate_mean"] <= oil_rate_q995].copy()

print("Скважин для анализа:", wells.shape[0])
wells.head()

## 5. Leakage-free target

In [ ]:
wells["productivity_score"] = (
    np.log1p(wells["oil_per_month"]) * 0.55 +
    np.log1p(wells["oil_mass_rate_mean"]) * 0.35 -
    wells["wcut_rate_mean"] * 0.004
)

score_threshold = wells["productivity_score"].quantile(0.75)
wells["good_well"] = (wells["productivity_score"] >= score_threshold).astype(int)

print("Порог good_well:", score_threshold)
print(wells["good_well"].value_counts())
print(wells["good_well"].value_counts(normalize=True))

In [ ]:
wells[[
    "well_id","x","y","block","horizon","well_category",
    "oil_per_month","oil_mass_rate_mean","wcut_rate_mean",
    "productivity_score","good_well"
]].sort_values("productivity_score", ascending=False).head(20)

## 6. Spatial clusters для честной spatial CV

In [ ]:
wells, spatial_kmeans = add_spatial_clusters(wells, n_clusters=5, random_state=RANDOM_STATE)
wells[["well_id", "x", "y", "spatial_cluster"]].head()

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=wells,
    x="x",
    y="y",
    hue="spatial_cluster",
    palette="tab10",
    s=15,
    alpha=0.65
)
plt.title("Spatial clusters for CV")
plt.show()

## 7. Neighbor features

In [ ]:
def build_neighbor_features(
    points_df,
    reference_df,
    exclude_self=False,
    max_k=15,
    radii_km=(0.5, 1.0, 1.5, 2.0, 3.0)
):
    points_rad = prepare_coords_rad(points_df)
    ref_rad = prepare_coords_rad(reference_df)

    n_neighbors = min(max_k + 1 if exclude_self else max_k, len(reference_df))
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric="haversine")
    nn.fit(ref_rad)

    distances_rad, indices = nn.kneighbors(points_rad)
    distances_km = distances_rad * EARTH_RADIUS_KM

    if exclude_self:
        distances_km = distances_km[:, 1:]
        indices = indices[:, 1:]

    ref = reference_df.reset_index(drop=True)

    ref_oil = ref["oil_per_month"].to_numpy()
    ref_rate = ref["oil_mass_rate_mean"].to_numpy()
    ref_wcut = ref["wcut_rate_mean"].to_numpy()
    ref_score = ref["productivity_score"].to_numpy()
    ref_good = ref["good_well"].to_numpy()
    ref_months = ref["months_active"].to_numpy()

    feat = pd.DataFrame(index=points_df.index)

    feat["dist_nearest_km"] = distances_km[:, 0]

    for k in [3, 5, 10, 15]:
        kk = min(k, indices.shape[1])

        idx_k = indices[:, :kk]
        dist_k = distances_km[:, :kk]

        feat[f"dist_mean_{k}_km"] = np.nanmean(dist_k, axis=1)
        feat[f"dist_min_{k}_km"] = np.nanmin(dist_k, axis=1)
        feat[f"dist_max_{k}_km"] = np.nanmax(dist_k, axis=1)

        feat[f"near_{k}_oil_mean"] = np.nanmean(ref_oil[idx_k], axis=1)
        feat[f"near_{k}_oil_max"] = np.nanmax(ref_oil[idx_k], axis=1)
        feat[f"near_{k}_oil_median"] = np.nanmedian(ref_oil[idx_k], axis=1)

        feat[f"near_{k}_rate_mean"] = np.nanmean(ref_rate[idx_k], axis=1)
        feat[f"near_{k}_rate_max"] = np.nanmax(ref_rate[idx_k], axis=1)

        feat[f"near_{k}_wcut_mean"] = np.nanmean(ref_wcut[idx_k], axis=1)
        feat[f"near_{k}_wcut_min"] = np.nanmin(ref_wcut[idx_k], axis=1)

        feat[f"near_{k}_score_mean"] = np.nanmean(ref_score[idx_k], axis=1)
        feat[f"near_{k}_score_max"] = np.nanmax(ref_score[idx_k], axis=1)
        feat[f"near_{k}_good_ratio"] = np.nanmean(ref_good[idx_k], axis=1)

        feat[f"near_{k}_months_mean"] = np.nanmean(ref_months[idx_k], axis=1)

    nearest_idx = indices[:, 0]
    feat["nearest_block"] = ref.loc[nearest_idx, "block"].astype(str).to_numpy()
    feat["nearest_horizon"] = ref.loc[nearest_idx, "horizon"].astype(str).to_numpy()
    feat["nearest_well_category"] = ref.loc[nearest_idx, "well_category"].astype(str).to_numpy()

    for radius in radii_km:
        radius_rad = radius / EARTH_RADIUS_KM

        neigh_dist, neigh_ind = nn.radius_neighbors(
            points_rad,
            radius=radius_rad,
            return_distance=True
        )

        counts = []
        oil_means = []
        rate_means = []
        wcut_means = []
        good_ratios = []

        for d_arr, i_arr in zip(neigh_dist, neigh_ind):
            if exclude_self:
                mask = d_arr > 1e-10
                i_arr = i_arr[mask]

            counts.append(len(i_arr))

            if len(i_arr) == 0:
                oil_means.append(np.nan)
                rate_means.append(np.nan)
                wcut_means.append(np.nan)
                good_ratios.append(np.nan)
            else:
                oil_means.append(np.nanmean(ref_oil[i_arr]))
                rate_means.append(np.nanmean(ref_rate[i_arr]))
                wcut_means.append(np.nanmean(ref_wcut[i_arr]))
                good_ratios.append(np.nanmean(ref_good[i_arr]))

        rname = str(radius).replace(".", "_")
        feat[f"radius_{rname}_km_count"] = counts
        feat[f"radius_{rname}_km_oil_mean"] = oil_means
        feat[f"radius_{rname}_km_rate_mean"] = rate_means
        feat[f"radius_{rname}_km_wcut_mean"] = wcut_means
        feat[f"radius_{rname}_km_good_ratio"] = good_ratios

    return feat

## 8. Общие функции оценки моделей

In [ ]:
model_results = []

def evaluate_classifier(model_name, y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)

    row = {
        "model": model_name,
        "roc_auc": roc_auc_score(y_true, proba),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "threshold": threshold
    }
    model_results.append(row)

    print(model_name)
    print("=" * 60)
    print("ROC-AUC :", row["roc_auc"])
    print("Precision:", row["precision"])
    print("Recall   :", row["recall"])
    print("F1       :", row["f1"])
    print()
    print(classification_report(y_true, pred, zero_division=0))

    cm = confusion_matrix(y_true, pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion matrix — {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return row

def zscore_with_train_stats(series, mean_value, std_value):
    if std_value == 0 or pd.isna(std_value):
        return series * 0
    return (series - mean_value) / std_value

def build_knn_baseline_score(X, stats=None):
    raw = pd.DataFrame(index=X.index)
    raw["oil"] = X["near_5_oil_mean"]
    raw["rate"] = X["near_5_rate_mean"]
    raw["good_ratio"] = X["near_5_good_ratio"]
    raw["score"] = X["near_5_score_mean"]
    raw["wcut"] = X["near_5_wcut_mean"]
    raw["distance"] = X["dist_nearest_km"]

    if stats is None:
        stats = {}
        for col in raw.columns:
            stats[col] = {"mean": raw[col].mean(), "std": raw[col].std()}

    oil_z = zscore_with_train_stats(raw["oil"], stats["oil"]["mean"], stats["oil"]["std"])
    rate_z = zscore_with_train_stats(raw["rate"], stats["rate"]["mean"], stats["rate"]["std"])
    score_z = zscore_with_train_stats(raw["score"], stats["score"]["mean"], stats["score"]["std"])
    wcut_z = zscore_with_train_stats(raw["wcut"], stats["wcut"]["mean"], stats["wcut"]["std"])
    dist_z = zscore_with_train_stats(raw["distance"], stats["distance"]["mean"], stats["distance"]["std"])

    final_score = (
        0.30 * oil_z +
        0.25 * rate_z +
        0.25 * raw["good_ratio"].fillna(0) +
        0.20 * score_z -
        0.15 * wcut_z -
        0.05 * dist_z
    )

    return final_score, stats

## 9. Один spatial holdout для многомодельного сравнения

In [ ]:
# Берем один spatial holdout: 4 кластера train, 1 кластер test
test_cluster = wells["spatial_cluster"].value_counts().sort_index().index[-1]

train_wells = wells[wells["spatial_cluster"] != test_cluster].copy()
test_wells = wells[wells["spatial_cluster"] == test_cluster].copy()

print("Test cluster:", test_cluster)
print("Train wells:", train_wells.shape)
print("Test wells :", test_wells.shape)
print("Train good ratio:", train_wells["good_well"].mean())
print("Test good ratio :", test_wells["good_well"].mean())

In [ ]:
X_train_raw = build_neighbor_features(
    points_df=train_wells,
    reference_df=train_wells,
    exclude_self=True
)

X_test_raw = build_neighbor_features(
    points_df=test_wells,
    reference_df=train_wells,
    exclude_self=False
)

y_train = train_wells["good_well"].copy()
y_test = test_wells["good_well"].copy()

y_train_reg = train_wells["productivity_score"].copy()
y_test_reg = test_wells["productivity_score"].copy()

categorical_features = ["nearest_block", "nearest_horizon", "nearest_well_category"]
numeric_features = [c for c in X_train_raw.columns if c not in categorical_features]

X_train = X_train_raw.copy()
X_test = X_test_raw.copy()

for col in categorical_features:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

fill_values = {}
for col in numeric_features:
    med = X_train[col].median()
    fill_values[col] = med
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
X_train.head()

## 10. Модуль A — KNN Spatial Baseline

In [ ]:
knn_train_score, knn_stats = build_knn_baseline_score(X_train)
knn_test_score, _ = build_knn_baseline_score(X_test, stats=knn_stats)

knn_test_proba = knn_test_score.rank(pct=True).to_numpy()
knn_threshold = np.quantile(knn_test_proba, 1 - y_train.mean())

evaluate_classifier(
    "KNN Spatial Scoring Baseline",
    y_test,
    knn_test_proba,
    threshold=knn_threshold
)

## 11. Модуль B — Logistic Regression

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

log_reg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

log_reg_model.fit(X_train, y_train)
log_reg_proba = log_reg_model.predict_proba(X_test)[:, 1]

evaluate_classifier(
    "Logistic Regression Baseline",
    y_test,
    log_reg_proba,
    threshold=0.5
)

## 12. Модуль C — Random Forest

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

evaluate_classifier(
    "Random Forest Location Classifier",
    y_test,
    rf_proba,
    threshold=0.5
)

## 13. Модуль D — LightGBM

In [ ]:
# Для LGBM/XGB закодируем категории простыми category codes
def encode_for_boosting(df_in, cat_features):
    out = df_in.copy()
    mappings = {}
    for col in cat_features:
        out[col] = out[col].astype("category")
        mappings[col] = list(out[col].cat.categories)
        out[col] = out[col].cat.codes
    return out, mappings

X_train_boost, boost_mappings = encode_for_boosting(X_train, categorical_features)
X_test_boost = X_test.copy()

for col in categorical_features:
    categories = boost_mappings[col]
    X_test_boost[col] = pd.Categorical(X_test_boost[col], categories=categories).codes

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    verbose=-1
)

lgbm_model.fit(X_train_boost, y_train)
lgbm_proba = lgbm_model.predict_proba(X_test_boost)[:, 1]

evaluate_classifier(
    "LightGBM Location Classifier",
    y_test,
    lgbm_proba,
    threshold=0.5
)

## 14. Модуль E — XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_model.fit(X_train_boost, y_train)
xgb_proba = xgb_model.predict_proba(X_test_boost)[:, 1]

evaluate_classifier(
    "XGBoost Location Classifier",
    y_test,
    xgb_proba,
    threshold=0.5
)

## 15. Модуль F — CatBoost Classifier

In [ ]:
cat_features_indices = [X_train.columns.get_loc(c) for c in categorical_features]

catboost_classifier = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=100,
    allow_writing_files=False
)

catboost_classifier.fit(
    X_train,
    y_train,
    cat_features=cat_features_indices,
    eval_set=(X_test, y_test),
    use_best_model=True
)

catboost_proba = catboost_classifier.predict_proba(X_test)[:, 1]

evaluate_classifier(
    "CatBoost Location Classifier",
    y_test,
    catboost_proba,
    threshold=0.5
)

## 16. Модуль G — CatBoost Regressor as Ranker

In [ ]:
catboost_regressor = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RANDOM_STATE,
    verbose=100,
    allow_writing_files=False
)

catboost_regressor.fit(
    X_train,
    y_train_reg,
    cat_features=cat_features_indices,
    eval_set=(X_test, y_test_reg),
    use_best_model=True
)

reg_pred_score = catboost_regressor.predict(X_test)

reg_mae = mean_absolute_error(y_test_reg, reg_pred_score)
reg_rmse = mean_squared_error(y_test_reg, reg_pred_score) ** 0.5
reg_r2 = r2_score(y_test_reg, reg_pred_score)

print("CatBoost Productivity Regressor")
print("=" * 60)
print("MAE :", reg_mae)
print("RMSE:", reg_rmse)
print("R2  :", reg_r2)

In [ ]:
reg_proba_like = pd.Series(reg_pred_score).rank(pct=True).to_numpy()
reg_threshold = np.quantile(reg_proba_like, 1 - y_train.mean())

evaluate_classifier(
    "CatBoost Regressor as Ranker",
    y_test,
    reg_proba_like,
    threshold=reg_threshold
)

## 17. Сравнение всех модулей

In [ ]:
model_results_df = pd.DataFrame(model_results).sort_values("roc_auc", ascending=False)
model_results_df

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=model_results_df, x="model", y="roc_auc")
plt.title("Model comparison by ROC-AUC")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=model_results_df, x="model", y="f1")
plt.title("Model comparison by F1")
plt.xticks(rotation=45, ha="right")
plt.show()

## 18. Честная spatial CV только для CatBoost (основной модуль)

In [ ]:
gkf = GroupKFold(n_splits=5)
cv_rows = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(wells, groups=wells["spatial_cluster"]), start=1):
    tr = wells.iloc[tr_idx].copy()
    va = wells.iloc[va_idx].copy()

    X_tr = build_neighbor_features(tr, tr, exclude_self=True)
    X_va = build_neighbor_features(va, tr, exclude_self=False)

    y_tr_cls = tr["good_well"].copy()
    y_va_cls = va["good_well"].copy()

    y_tr_reg = tr["productivity_score"].copy()
    y_va_reg = va["productivity_score"].copy()

    for col in categorical_features:
        X_tr[col] = X_tr[col].astype(str)
        X_va[col] = X_va[col].astype(str)

    num_cols_cv = [c for c in X_tr.columns if c not in categorical_features]
    for col in num_cols_cv:
        med = X_tr[col].median()
        X_tr[col] = X_tr[col].fillna(med)
        X_va[col] = X_va[col].fillna(med)

    cat_idx_cv = [X_tr.columns.get_loc(c) for c in categorical_features]

    clf_cv = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False
    )

    reg_cv = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False
    )

    clf_cv.fit(X_tr, y_tr_cls, cat_features=cat_idx_cv)
    reg_cv.fit(X_tr, y_tr_reg, cat_features=cat_idx_cv)

    va_proba = clf_cv.predict_proba(X_va)[:, 1]
    va_pred = (va_proba >= 0.5).astype(int)
    va_reg = reg_cv.predict(X_va)

    cv_rows.append({
        "fold": fold,
        "cls_roc_auc": roc_auc_score(y_va_cls, va_proba),
        "cls_precision": precision_score(y_va_cls, va_pred, zero_division=0),
        "cls_recall": recall_score(y_va_cls, va_pred, zero_division=0),
        "cls_f1": f1_score(y_va_cls, va_pred, zero_division=0),
        "reg_mae": mean_absolute_error(y_va_reg, va_reg),
        "reg_rmse": mean_squared_error(y_va_reg, va_reg) ** 0.5,
        "reg_r2": r2_score(y_va_reg, va_reg),
    })

cv_results = pd.DataFrame(cv_rows)
cv_results

In [ ]:
cv_results.mean(numeric_only=True)

## 19. Обучение финальных моделей на всех скважинах

In [ ]:
X_all = build_neighbor_features(
    points_df=wells,
    reference_df=wells,
    exclude_self=True
)

for col in categorical_features:
    X_all[col] = X_all[col].astype(str)

numeric_features_all = [c for c in X_all.columns if c not in categorical_features]

fill_values_all = {}
for col in numeric_features_all:
    med = X_all[col].median()
    fill_values_all[col] = med
    X_all[col] = X_all[col].fillna(med)

cat_features_indices_all = [X_all.columns.get_loc(c) for c in categorical_features]

final_catboost_classifier = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=100,
    allow_writing_files=False
)

final_catboost_regressor = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RANDOM_STATE,
    verbose=100,
    allow_writing_files=False
)

final_catboost_classifier.fit(
    X_all, wells["good_well"], cat_features=cat_features_indices_all
)

final_catboost_regressor.fit(
    X_all, wells["productivity_score"], cat_features=cat_features_indices_all
)

## 20. Важность признаков

In [ ]:
catboost_importance = pd.DataFrame({
    "feature": X_all.columns,
    "importance": final_catboost_classifier.get_feature_importance()
}).sort_values("importance", ascending=False)

catboost_importance.head(30)

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(
    data=catboost_importance.head(25),
    x="importance",
    y="feature"
)
plt.title("CatBoost Classifier — top-25 features")
plt.show()

## 21. KDE support zone вместо convex hull

In [ ]:
x_vals = wells["x"].to_numpy()
y_vals = wells["y"].to_numpy()

grid_step_km = 0.5

mean_lat = wells["y"].mean()
lat_step = grid_step_km / 111.0
lon_step = grid_step_km / (111.0 * np.cos(np.radians(mean_lat)))

x_min, x_max = wells["x"].min(), wells["x"].max()
y_min, y_max = wells["y"].min(), wells["y"].max()

x_grid = np.arange(x_min, x_max, lon_step)
y_grid = np.arange(y_min, y_max, lat_step)

XX, YY = np.meshgrid(x_grid, y_grid)

candidate_points = pd.DataFrame({
    "x": XX.ravel(),
    "y": YY.ravel()
})

print("Всего точек сетки:", candidate_points.shape)
candidate_points.head()

In [ ]:
kde = gaussian_kde(np.vstack([x_vals, y_vals]))
candidate_points["kde_density"] = kde(np.vstack([candidate_points["x"], candidate_points["y"]]))

density_threshold = np.percentile(candidate_points["kde_density"], 10)
candidate_points["inside_support_zone"] = candidate_points["kde_density"] >= density_threshold

print("Точек внутри support zone:", candidate_points["inside_support_zone"].sum())

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(
    candidate_points["x"],
    candidate_points["y"],
    c=candidate_points["kde_density"],
    s=8,
    cmap="viridis"
)
plt.colorbar(label="KDE density")
plt.scatter(wells["x"], wells["y"], s=4, alpha=0.25, color="white")
plt.title("KDE support zone")
plt.show()

## 22. Признаки для кандидатных пустых точек

In [ ]:
candidate_points = candidate_points[candidate_points["inside_support_zone"]].copy()

candidate_features = build_neighbor_features(
    points_df=candidate_points,
    reference_df=wells,
    exclude_self=False
)

for col in categorical_features:
    candidate_features[col] = candidate_features[col].astype(str)

for col in numeric_features_all:
    candidate_features[col] = candidate_features[col].fillna(fill_values_all[col])

candidate_points = pd.concat(
    [candidate_points.reset_index(drop=True), candidate_features.reset_index(drop=True)],
    axis=1
)

candidate_points.head()

## 23. Candidate Filtering Module

In [ ]:
candidate_filtered = candidate_points[
    (candidate_points["dist_nearest_km"] >= 0.30) &
    (candidate_points["dist_nearest_km"] <= 3.00) &
    (candidate_points["radius_1_0_km_count"] >= 3) &
    (candidate_points["radius_1_5_km_count"] >= 5)
].copy()

print("После геометрических фильтров:", candidate_filtered.shape)

In [ ]:
candidate_filtered = candidate_filtered[
    candidate_filtered["near_5_wcut_mean"].fillna(100) <= 85
].copy()

print("После water-cut filter:", candidate_filtered.shape)
candidate_filtered.head()

## 24. Ranking Module

In [ ]:
X_candidates = candidate_filtered[X_all.columns].copy()

candidate_knn_score, _ = build_knn_baseline_score(X_candidates, stats=knn_stats)
candidate_filtered["knn_score"] = candidate_knn_score
candidate_filtered["knn_score_norm"] = candidate_filtered["knn_score"].rank(pct=True)

candidate_filtered["catboost_good_probability"] = (
    final_catboost_classifier.predict_proba(X_candidates)[:, 1]
)

candidate_filtered["catboost_predicted_productivity"] = (
    final_catboost_regressor.predict(X_candidates)
)

candidate_filtered["catboost_predicted_productivity_norm"] = (
    candidate_filtered["catboost_predicted_productivity"].rank(pct=True)
)

candidate_filtered["neighbor_good_ratio_norm"] = (
    candidate_filtered["near_5_good_ratio"].fillna(0).rank(pct=True)
)

candidate_filtered["distance_penalty"] = 0.0
candidate_filtered.loc[candidate_filtered["dist_nearest_km"] < 0.5, "distance_penalty"] = 0.10
candidate_filtered.loc[candidate_filtered["dist_nearest_km"] > 2.5, "distance_penalty"] = 0.08

candidate_filtered["water_penalty"] = 0.0
candidate_filtered.loc[candidate_filtered["near_5_wcut_mean"] > 80, "water_penalty"] = 0.12

candidate_filtered["final_location_score"] = (
    0.40 * candidate_filtered["catboost_good_probability"] +
    0.25 * candidate_filtered["catboost_predicted_productivity_norm"] +
    0.20 * candidate_filtered["knn_score_norm"] +
    0.15 * candidate_filtered["neighbor_good_ratio_norm"] -
    candidate_filtered["distance_penalty"] -
    candidate_filtered["water_penalty"]
)

top_locations = (
    candidate_filtered
    .sort_values("final_location_score", ascending=False)
    .head(100)
    .copy()
)

top_locations[[
    "x","y","final_location_score",
    "catboost_good_probability",
    "catboost_predicted_productivity",
    "knn_score_norm",
    "dist_nearest_km",
    "near_3_oil_mean",
    "near_5_oil_mean",
    "near_5_rate_mean",
    "near_5_wcut_mean",
    "near_5_good_ratio",
    "radius_1_0_km_count",
    "nearest_block",
    "nearest_horizon"
]].head(30)

## 25. Лучшая точка

In [ ]:
best_point = top_locations.iloc[0].copy()

print("=" * 60)
print("ЛУЧШАЯ РЕКОМЕНДОВАННАЯ ТОЧКА")
print("=" * 60)
print(f"x: {best_point['x']:.5f}")
print(f"y: {best_point['y']:.5f}")
print(f"final_location_score: {best_point['final_location_score']:.4f}")
print(f"catboost_good_probability: {best_point['catboost_good_probability']:.4f}")
print(f"catboost_predicted_productivity: {best_point['catboost_predicted_productivity']:.4f}")
print(f"dist_nearest_km: {best_point['dist_nearest_km']:.3f}")
print(f"near_5_oil_mean: {best_point['near_5_oil_mean']:.3f}")
print(f"near_5_rate_mean: {best_point['near_5_rate_mean']:.3f}")
print(f"near_5_wcut_mean: {best_point['near_5_wcut_mean']:.3f}")
print(f"nearest_block: {best_point['nearest_block']}")
print(f"nearest_horizon: {best_point['nearest_horizon']}")
print("=" * 60)

## 26. Ближайшие реальные соседи к лучшей точке

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

neighbors = wells.copy()
neighbors["dist_to_best_km"] = haversine_km(
    best_point["y"], best_point["x"],
    neighbors["y"], neighbors["x"]
)

nearest_neighbors = neighbors.nsmallest(10, "dist_to_best_km")[[
    "well_id","x","y","dist_to_best_km",
    "oil_per_month","oil_mass_rate_mean","wcut_rate_mean",
    "months_active","block","horizon","good_well"
]]

nearest_neighbors

## 27. Карта рекомендаций

In [ ]:
plt.figure(figsize=(13, 10))

plt.scatter(
    wells["x"], wells["y"],
    s=8, alpha=0.25, label="Existing wells"
)

good_existing = wells[wells["good_well"] == 1]
plt.scatter(
    good_existing["x"], good_existing["y"],
    s=12, alpha=0.55, label="Good wells"
)

plt.scatter(
    top_locations.head(20)["x"],
    top_locations.head(20)["y"],
    s=110,
    c=top_locations.head(20)["final_location_score"],
    cmap="autumn",
    edgecolors="black",
    label="Top-20 candidate locations"
)

plt.scatter(
    best_point["x"], best_point["y"],
    s=350, marker="*",
    c="red", edgecolors="darkred", linewidths=2,
    label="Best location"
)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("v3 — Multi-model recommended candidate locations")
plt.legend()
plt.show()

## 28. Сохранение результатов

In [ ]:
output_dir = Path("location_v3_results")
output_dir.mkdir(exist_ok=True)

# Таблицы
wells.to_csv(output_dir / "well_base.csv", index=False)
model_results_df.to_csv(output_dir / "model_comparison.csv", index=False)
cv_results.to_csv(output_dir / "spatial_cv_results.csv", index=False)
catboost_importance.to_csv(output_dir / "catboost_feature_importance.csv", index=False)
candidate_filtered.to_csv(output_dir / "all_candidate_locations_scored.csv", index=False)
top_locations.to_csv(output_dir / "top_candidate_locations.csv", index=False)
nearest_neighbors.to_csv(output_dir / "nearest_neighbors_to_best_point.csv", index=False)

# Модели
final_catboost_classifier.save_model(output_dir / "catboost_location_classifier_v3.cbm")
final_catboost_regressor.save_model(output_dir / "catboost_productivity_regressor_v3.cbm")
joblib.dump(log_reg_model, output_dir / "logistic_regression_baseline.joblib")
joblib.dump(rf_model, output_dir / "random_forest_classifier.joblib")
joblib.dump(lgbm_model, output_dir / "lightgbm_classifier.joblib")
joblib.dump(xgb_model, output_dir / "xgboost_classifier.joblib")

# Excel
excel_path = output_dir / "location_v3_full_report.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    model_results_df.to_excel(writer, sheet_name="model_comparison", index=False)
    cv_results.to_excel(writer, sheet_name="spatial_cv", index=False)
    catboost_importance.head(50).to_excel(writer, sheet_name="feature_importance", index=False)
    top_locations.head(100).to_excel(writer, sheet_name="top_locations", index=False)
    nearest_neighbors.to_excel(writer, sheet_name="nearest_neighbors", index=False)

print("Все результаты сохранены в папку:")
print(output_dir.resolve())